# Exploratory Data Analysis (EDA) — Real-World Resume & Job Description Dataset
## Course: AI & Machine Learning | 2nd Year Project

This notebook conducts a complete Exploratory Data Analysis (EDA) on real-world applicant resumes (`.docx`, `.txt`) across three technical career domains:
1. **Java & J2EE / Full-Stack Engineering**
2. **IT Business Analysis & Agile Scrum**
3. **Backend Python Engineering**

### Step 1: Import Libraries and Load Data Loader

In [1]:
import os
import re
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import data_loader

print("All libraries imported successfully!")

### Step 2: Ingest Real Candidate Resumes from `sample_data/`

In [2]:
sample_dir = "sample_data"

# Find all candidate resumes
files = [f for f in sorted(os.listdir(sample_dir)) 
         if f.lower().endswith(('.docx', '.txt')) 
         and not f.startswith('job_description') 
         and not f.startswith('test_empty')]

corpus = []
for f in files:
    path = os.path.join(sample_dir, f)
    text = data_loader.load_text_from_file(path)
    words = re.findall(r'\b[a-zA-Z]{2,}\b', text.lower())
    corpus.append({
        "filename": f,
        "char_count": len(text),
        "word_count": len(words),
        "unique_words": len(set(words)),
        "lexical_diversity": round(len(set(words)) / len(words), 4) if words else 0,
        "words": words,
        "text": text
    })

print(f"Successfully ingested {len(corpus)} applicant documents.")

### Step 3: Document Length & Lexical Diversity Table

In [3]:
df_stats = pd.DataFrame([{
    "Resume Document": c["filename"],
    "Total Characters": c["char_count"],
    "Total Words": c["word_count"],
    "Unique Words": c["unique_words"],
    "Lexical Diversity": c["lexical_diversity"]
} for c in corpus])

df_stats

### Step 4: Visualizing Document Length Distribution
Demonstrating the massive variance between concise summaries (~100 words) and multi-page dossiers (up to 6,300+ words).

In [4]:
plt.figure(figsize=(10, 5))
clean_names = [c["filename"].replace('.docx', '').replace('.DOCX', '').replace('.txt', '')[:16] for c in corpus]
word_counts = [c["word_count"] for c in corpus]

bars = plt.barh(clean_names, word_counts, color="#3498db", edgecolor="#2980b9", height=0.65)
plt.title("EDA: Candidate Resume Word Count Distribution", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Word Count", fontsize=11)
plt.ylabel("Candidate Profile", fontsize=11)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 50, bar.get_y() + bar.get_height()/2, f"{int(w):,}", va="center", fontsize=8.5, color="#2c3e50")

plt.xlim(0, max(word_counts) * 1.15)
plt.tight_layout()
plt.show()

### Step 5: Technical Skill Term Frequency Analysis
Extracting occurrences of key technical competencies across the candidates.

In [5]:
tracked_skills = ["python", "java", "spring", "aws", "sql", "docker", "agile", "scrum", "microservices", "hadoop", "spark", "jira"]

skill_records = []
for c in corpus:
    counts = Counter(c["words"])
    row = {s: counts[s] for s in tracked_skills}
    row["Candidate"] = c["filename"].replace('.docx', '').replace('.DOCX', '').replace('.txt', '')[:16]
    skill_records.append(row)

df_skills = pd.DataFrame(skill_records).set_index("Candidate")
df_skills

### Step 6: Technical Competencies Heatmap
Visualizing the domain clusters: Java Engineers, Business Analysts, and Big Data Developers.

In [6]:
plt.figure(figsize=(11, 6))
sns.heatmap(df_skills, cmap="YlGnBu", annot=True, fmt="d", cbar=True, linewidths=0.5)
plt.title("EDA: Technical Skill Term Frequency Heatmap", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Technical Skills", fontsize=11)
plt.ylabel("Candidate Profile", fontsize=11)
plt.tight_layout()
plt.show()

### Step 7: Key EDA Findings for Project Report & PPT
1. **Document Length Variance**: Real resumes range from 14 words (test stub) and 102 words (concise summary) to 6,332 words (Priya B.docx, 9 pages).
2. **Lexical Richness vs. Length**: As resume length expands with redundant project descriptions, lexical diversity drops from ~80% down to 13%.
3. **Domain Clustering**:
   - **Java/Spring Cluster**: Priya (77 Java / 93 Spring), sai k (45/45), Sai Srinivas (40/18), Varun (31/58), Vishnu (22/22).
   - **Business Analysis Cluster**: Vikash & Teresa (dominant in Scrum, Agile, Jira, and BRDs).
   - **Big Data Cluster**: Vijay Bhargav (Hadoop: 26, Spark: 24).
   - **Backend Python Cluster**: resume.txt (FastAPI, Python backend).